In [ ]:
import warnings

warnings.filterwarnings('ignore')

### Установим красивые дефолтные настройки
### Может быть лень постоянно прописывать
### У графиков параметры цвета, размера, шрифта
### Можно положить их в словарь дефолтных настроек

import matplotlib as mlp

mlp.rcParams['lines.linewidth'] = 5
mlp.rcParams['xtick.major.size'] = 20
mlp.rcParams['xtick.major.width'] = 5
mlp.rcParams['xtick.labelsize'] = 20
mlp.rcParams['xtick.color'] = '#FF5533'

mlp.rcParams['ytick.major.size'] = 20
mlp.rcParams['ytick.major.width'] = 5
mlp.rcParams['ytick.labelsize'] = 20
mlp.rcParams['ytick.color'] = '#FF5533'

mlp.rcParams['axes.labelsize'] = 20
mlp.rcParams['axes.titlesize'] = 20
mlp.rcParams['axes.titlecolor'] = '#00B050'
mlp.rcParams['axes.labelcolor'] = '#00B050'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.options.display.max_columns = 500

df = pd.read_csv("banking.csv")

In [ ]:
df.head()

In [ ]:
df.shape

<dl>
<dt> Описание колонок:
<dd>age - возраст клиента </dd>
<dd>job - тип работы </dd>
<dd>marital - статус замужества</dd>
<dd>education - степень образования </dd>
<dd>default - дефолтил ли клиент? </dd>
<dd>housing - есть ли жилищный кредит? </dd>
<dd>loan - есть ли потребительский кредит? </dd>
<dd>contact - тип коммуникации </dd>
<dd>month - месяц последнего контакта </dd>
<dd>day_of_week - день недели последнего контакта </dd>
<dd>duration - длительность последнего звонка - стоит убрать при обучении модели </dd>
<dd>campaign - количество звонков в течение маркетинговой кампании </dd>
<dd>pdays - как давно не было коммуникации с клиентом, относительно предыдущей маркетинговой кампании </dd>
<dd>previous - количество звоноков до текущей маркетинговой компании </dd>
<dd>poutcome - результат предыдущей маркентинговой кампании относительно выбранного клиента </dd>
<dd>emp.var.rate - коэффициент вариации безработицы (quarterly) </dd>
<dd>cons.price.idx - индекс потребительских цен (monthly) </dd>
<dd>cons.conf.idx: - индекс потребительской уверенности (monthly) </dd>
<dd>euribor3m - межбанковская европейская ставка предложения по трехмесячному займу (daily) </dd>
<dd>nr.employed - количество занятых </dd>
<dd>y - таргетная переменная: взял ли клиент депозит </dd>
</dl>

**Наша целевая переменная - взял ли клиент депозит.**

In [ ]:
df = df.drop('duration', axis=1)

In [ ]:
numeric_columns = df.loc[:, df.dtypes != np.object_].columns
df.loc[:, df.dtypes != np.object_].head()

In [ ]:
categorical_columns = df.loc[:, df.dtypes == np.object_].columns
df.loc[:, df.dtypes == np.object_].head()

In [ ]:
df.isna().sum()

In [ ]:
df.describe()

In [ ]:
df[numeric_columns].corr()

In [ ]:
import seaborn as sns

fig = plt.figure()
fig.set_size_inches(16, 10)
sns.heatmap(df[numeric_columns].corr(), xticklabels=numeric_columns, yticklabels=numeric_columns, cmap='BrBG', vmin=-1,
            vmax=1)


In [ ]:
### Секретные функции для фильтрации признаков

def get_redundant_pairs(df):
    pairs_to_drop = set()
    cols = df.columns
    for i in range(0, df.shape[1]):
        for j in range(0, i + 1):
            pairs_to_drop.add((cols[i], cols[j]))
    return pairs_to_drop


def get_top_abs_correlations(df, n=5):
    au_corr = df.corr().abs().unstack()
    labels_to_drop = get_redundant_pairs(df)
    au_corr = au_corr.drop(labels=labels_to_drop).sort_values(ascending=False)
    return au_corr[0:n]


print("Top Absolute Correlations")
print(get_top_abs_correlations(df[numeric_columns], 10))

In [ ]:
test = df[numeric_columns].drop(['emp_var_rate', 'euribor3m'], axis=1)
get_top_abs_correlations(test, 10)

In [ ]:
df = df.drop(['emp_var_rate', 'euribor3m'], axis=1)
numeric_columns = numeric_columns.drop(['emp_var_rate', 'euribor3m'])

In [ ]:
### Посмотрим на квазиконстантые признаки

from sklearn.feature_selection import VarianceThreshold

cutter = VarianceThreshold(threshold=0.1)
cutter.fit(df[numeric_columns])

cutter.get_feature_names_out()

In [ ]:
numeric_columns

In [ ]:
### Еще один способ, как в задаче классификации без построения модели
### оценить важность вещественных признаков- с помощью ящиков с усами!
### Только теперь немного "наоборот", представляя таргет как категорию
### А значения, распределение которых хотим сравнивать, окажутся нашими фичами

num_col = ['age', 'nr_employed']

for col in num_col:
    fig = plt.figure()
    fig.set_size_inches(16, 10)

    sns.boxplot(y=col, x=df['y'].astype('category'), data=df)
    fig.show()



In [ ]:
### Посмотрим на распределение категорий среди разных таргетов

df.describe(include='object')

In [ ]:
### Гистограммы распределений в разных классах

for col in categorical_columns:
    g = sns.catplot(x=col, kind='count', col='y', data=df, sharey=False)
    g.set_xticklabels(rotation=60)

In [ ]:
df = df.drop(['loan', 'housing', 'marital'], axis=1)
categorical_columns = categorical_columns.drop(['loan', 'housing', 'marital'])

In [ ]:
df.head().shape

### Закодируем оставшиеся категориальные фичи!

In [ ]:
### Посмотрим, какие можно кодировать с помощью one-hot метода,
### а для каких лучше посчитать счетчики!

df.describe(include='object')

In [ ]:
for col in categorical_columns:

    ### К колонкам с маленькой размерностью применим one-hot
    if df[col].nunique() < 5:
        one_hot = pd.get_dummies(df[col], prefix=col, drop_first=True)
        df = pd.concat((df.drop(col, axis=1), one_hot), axis=1)

    ### К остальным - счетчики
    else:
        mean_target = df.groupby(col)['y'].mean()
        df[col] = df[col].map(mean_target)

In [ ]:
df.head()

In [ ]:
X = df.drop('y', axis=1)
Y = df['y']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y,
                                                    random_state=0,
                                                    test_size=0.2)

In [ ]:
X_train.shape[0], X_test.shape[0]

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

pipe = Pipeline([('scaler', StandardScaler()),
                 ('LR', LogisticRegression(penalty=None, solver='lbfgs', max_iter=1000))])

pipe.fit(X_train, Y_train)

In [ ]:
### Pipeline автоматом для классификации отображает accuracy

pipe.score(X_test, Y_test)

In [ ]:
X_test.head(1)

In [ ]:
### Чтобы предсказать вероятности соответственно классам
### Обратимся к аттрибуту classes_ и к методу predict_proba
### А чтобы понять уверенность модели, воспользуемся методом
### decision_function

print(pipe.classes_)

print(pipe.predict(X_test.head(1)))

print(pipe.predict_proba(X_test.head(1)))

print(pipe.decision_function(X_test.head(2)))

In [ ]:
### Константное предсказание на тесте

np.mean(Y_test == 0)

In [ ]:
df['y'].value_counts()

In [ ]:
from sklearn.metrics import confusion_matrix

tn, fp, fn, tp = confusion_matrix(Y_test, pipe.predict(X_test)).ravel()

print(f'True Negative errors: {tn}')
print(f'False Positive errors: {fp}')
print(f'False Negative errors: {fn}')
print(f'True Positive errors: {tp}')

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(Y_test, pipe.predict(X_test), labels=pipe.classes_)

cmp = ConfusionMatrixDisplay(confusion_matrix=cm)

fig, ax = plt.subplots(figsize=(16, 10))

cmp.plot(ax=ax)
plt.show()

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

print(f'accuracy = {accuracy_score(Y_test, pipe.predict(X_test)):.3f}')
print(f'precision = {precision_score(Y_test, pipe.predict(X_test)):.3f}')
print(f'recall = {recall_score(Y_test, pipe.predict(X_test)):.3f}')
print(f'f1 = {f1_score(Y_test, pipe.predict(X_test)):.3f}')

In [ ]:
pipe.predict_proba(X_test.head(1))[:, 1]

In [ ]:
from sklearn.metrics import precision_recall_curve

precision_recall_curve(Y_test, pipe.predict_proba(X_test)[:, 1])


In [ ]:
precision, recall, thresholds = precision_recall_curve(Y_test, pipe.predict_proba(X_test)[:, 1])

den = precision + recall
f_scores = np.where(den > 0, 2 * precision * recall / den, 0.0)
best_index = np.argmax(f_scores)
print("Best F-scope: ", f_scores[best_index])
print("thresholds for best F-scope: ", thresholds[best_index])

In [ ]:
precision[best_index]

In [ ]:
recall[best_index]

In [ ]:
threshold = thresholds[best_index]
y_pred = (pipe.predict_proba(X_test)[:,1] > threshold).astype('float')
y_pred

In [ ]:
cm = confusion_matrix(Y_test, y_pred, labels=pipe.classes_)

cmp = ConfusionMatrixDisplay(confusion_matrix=cm)

fig, ax = plt.subplots(figsize=(16, 10))

cmp.plot(ax=ax)
plt.show()

In [ ]:
from sklearn.metrics import roc_curve

frp, tpr, thresholds = roc_curve(Y_test, pipe.predict_proba(X_test)[:,1])

In [ ]:
from sklearn.metrics import RocCurveDisplay
RocCurveDisplay(fpr=frp, tpr=tpr).plot()

In [ ]:
from sklearn.metrics import auc

auc(frp, tpr)


In [ ]:
from sklearn.metrics import PrecisionRecallDisplay
precision, recall, thresholds = precision_recall_curve(Y_test, pipe.predict_proba(X_test)[:, 1])

PrecisionRecallDisplay(precision=precision, recall=recall).plot()

In [ ]:
auc(recall, precision)

In [ ]:
from sklearn.calibration import CalibrationDisplay

CalibrationDisplay.from_estimator(pipe, X_test, Y_test)

In [ ]:
pipe.predict_proba(X_test.head(1))

## SVM

In [ ]:
from sklearn.svm import LinearSVC

pipe_svm = Pipeline([("scaler_svm", StandardScaler()), ("SVM", LinearSVC())])

pipe_svm.fit(X_train, Y_train)

pipe_svm.score(X_test, Y_test)

In [ ]:
### Precision / Recall / F-мера

from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score


print(f'Accuracy равно: {accuracy_score(Y_test, pipe_svm.predict(X_test)):.3f}' )

print(f'Precision равно: {precision_score(Y_test, pipe_svm.predict(X_test)):.3f}' )

print(f'Recall равно: {recall_score(Y_test, pipe_svm.predict(X_test)):.3f}' )

print(f'F-мера равно: {f1_score(Y_test, pipe_svm.predict(X_test)):.3f}' )

In [ ]:
PrecisionRecallDisplay.from_estimator(pipe_svm, X_test, Y_test)

In [ ]:
RocCurveDisplay.from_estimator(pipe_svm, X_test, Y_test)

In [ ]:
### Изобразим калибровочную кривую
### Сейчас все сломается!

CalibrationDisplay.from_estimator(pipe_svm, X_test, Y_test)

In [ ]:
### Преобраузем выходы модели в вероятности
pipe_svm.decision_function(X_test.head(2))

def sigmoid(output):
    return 1 / (1 + np.exp(-output))

sigmoid(pipe_svm.decision_function(X_test))
pred_prob = sigmoid(pipe_svm.decision_function(X_test))

In [ ]:
min(pred_prob), max(pred_prob)

In [ ]:

CalibrationDisplay.from_predictions(Y_test, pred_prob, n_bins=15)

In [ ]:
### Калибровка Плата

from sklearn.calibration import CalibratedClassifierCV

calibration = CalibratedClassifierCV(pipe_svm, cv=5, method='sigmoid')
calibration.fit(X_train, Y_train)

calibrated_probs = calibration.predict_proba(X_test)[:, 1]

In [ ]:
calibrated_probs

In [ ]:
CalibrationDisplay.from_predictions(Y_test, calibrated_probs, n_bins=15)